In [1]:
import os
os.environ['ORT_DISABLE_TELEMETRY'] = '1'  # до любого импорта ORT/PyTorch-пайплайна
from pathlib import Path
import sys, subprocess, time
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Car-classification-MSK']
ROOT = next((p for p in candidates if (p / 'training/nive_mixed.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Открой notebook внутри Car-classification-MSK')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RUN_NAME = 'pilot_v1'
DEVICE = 'mps'  # на другой машине явно выбери cuda или cpu
started = time.perf_counter()
print('Python:', sys.executable, '\nПроект:', ROOT, '\nRun:', RUN_NAME, '| device:', DEVICE)


Python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Проект: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK 
Run: pilot_v1 | device: mps


In [2]:
print('Этап 1/5 — синтетические проверки trainer, resume и протокола', flush=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_nive_mixed.py', 'tests/test_nive_transfer.py', 'tests/test_overnight_training.py', 'tests/test_retrieval_policy.py'], check=True, env=os.environ.copy())


Этап 1/5 — синтетические проверки trainer, resume и протокола
........................................................................ [ 64%]
.......................................                                  [100%]
=============================== warnings summary ===============================
tests/test_nive_transfer.py::test_real_representation_transfer_and_onnx_export
  /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/training/osnet_ablation_suite.py:547: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
    torch.onnx.export(encoder, torch.zeros(1, 3, variant.size, variant.size), temporary,

tests/test_nive_tr

CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_nive_mixed.py', 'tests/test_nive_transfer.py', 'tests/test_overnight_training.py', 'tests/test_retrieval_policy.py'], returncode=0)

In [3]:
from training import nive_mixed as mixed
print('Этап 2/5 — проверка parent, train-only NiVe, дубликатов и защиты v25', flush=True)
context = mixed.prepare(RUN_NAME, device=DEVICE)
manifest = context['manifest']
print('Результаты:', context['output'])
print('Main LR:', manifest['config']['encoder_lr'], '| Head LR:', manifest['config']['encoder_lr'] * manifest['config']['head_lr_multiplier'])
print('NiVe:', manifest['nive']['used_images'], 'фото /', manifest['nive']['used_identities'], 'identity')
for arm, coverage in manifest['sampling']['planned_coverage'].items():
    print(arm, '| planned auxiliary coverage:', f"{coverage['unique_images']}/{coverage['available_images']}")


Этап 2/5 — проверка parent, train-only NiVe, дубликатов и защиты v25


NiVe integrity: train:   0%|          | 0/17070 [00:00<?, ?it/s]

NiVe integrity: test/query:   0%|          | 0/2887 [00:00<?, ?it/s]

NiVe integrity: test/gallery:   0%|          | 0/11778 [00:00<?, ?it/s]

AUDIT organizer: 500/9556
AUDIT organizer: 1000/9556
AUDIT organizer: 1500/9556
AUDIT organizer: 2000/9556
AUDIT organizer: 2500/9556
AUDIT organizer: 3000/9556
AUDIT organizer: 3500/9556
AUDIT organizer: 4000/9556
AUDIT organizer: 4500/9556
AUDIT organizer: 5000/9556
AUDIT organizer: 5500/9556
AUDIT organizer: 6000/9556
AUDIT organizer: 6500/9556
AUDIT organizer: 7000/9556
AUDIT organizer: 7500/9556
AUDIT organizer: 8000/9556
AUDIT organizer: 8500/9556
AUDIT organizer: 9000/9556
AUDIT organizer: 9500/9556
AUDIT NiVe train: 1000/17070
AUDIT NiVe train: 2000/17070
AUDIT NiVe train: 3000/17070
AUDIT NiVe train: 4000/17070
AUDIT NiVe train: 5000/17070
AUDIT NiVe train: 6000/17070
AUDIT NiVe train: 7000/17070
AUDIT NiVe train: 8000/17070
AUDIT NiVe train: 9000/17070
AUDIT NiVe train: 10000/17070
AUDIT NiVe train: 11000/17070
AUDIT NiVe train: 12000/17070
AUDIT NiVe train: 13000/17070
AUDIT NiVe train: 14000/17070
AUDIT NiVe train: 15000/17070
AUDIT NiVe train: 16000/17070
AUDIT NiVe train:

В `domain_audit.json` сохраняются яркость, blur, размеры, распределения identity/view, фиксированная случайная выборка и dHash-подозрения. Это не автоматическая разметка ночи/ошибочных bbox.
Если найдены cross-domain near-duplicate suspects, обучение останавливается **до** запуска N0/N1 и требует просмотра совпадений. Внутренние похожие кадры NiVe только описываются; исходные фотографии не удаляются.


In [4]:
print('Этап 3/5 — по три тестовых updates, проверка head-free ONNX', flush=True)
smoke = mixed.technical_smoke(context)
print('Smoke:', smoke['status'], '| настоящие N0/N1 начнутся с исходного parent')


Этап 3/5 — по три тестовых updates, проверка head-free ONNX


/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


Smoke: passed | настоящие N0/N1 начнутся с исходного parent


In [5]:
print('Этап 4/5 — полный парный пилот без ограничения времени', flush=True)
results = mixed.run_experiment(context)
print('Статус:', results['status'], '| screening decision:', results['decision'])


Этап 4/5 — полный парный пилот без ограничения времени
EVAL N_ref: fixed graph=0.844099, raw=0.832842
STAGE N0_target_extra: paired continuation; no original validation
TRAIN N0_target_extra: 25/1800 | joint | loss=2.9352 | elapsed 00:00:53
TRAIN N0_target_extra: 50/1800 | joint | loss=2.9587 | elapsed 00:01:44
TRAIN N0_target_extra: 75/1800 | joint | loss=3.0505 | elapsed 00:02:36
TRAIN N0_target_extra: 100/1800 | joint | loss=2.9648 | elapsed 00:03:27
TRAIN N0_target_extra: 125/1800 | joint | loss=2.8706 | elapsed 00:04:18
TRAIN N0_target_extra: 150/1800 | joint | loss=2.7523 | elapsed 00:05:08
TRAIN N0_target_extra: 175/1800 | joint | loss=2.7260 | elapsed 00:05:58
TRAIN N0_target_extra: 200/1800 | joint | loss=2.8082 | elapsed 00:06:48
TRAIN N0_target_extra: 225/1800 | joint | loss=2.7546 | elapsed 00:07:38
TRAIN N0_target_extra: 250/1800 | joint | loss=2.5977 | elapsed 00:08:28
TRAIN N0_target_extra: 275/1800 | joint | loss=2.6086 | elapsed 00:09:18
TRAIN N0_target_extra: 300/1800

In [6]:
from IPython.display import Markdown, display
print('Этап 5/5 — итоговый отчёт | время Run All:', round((time.perf_counter()-started)/60, 1), 'мин')
print('Выбранные updates:', results['selection'])
print('Разница N1 − N0:', results['N1_vs_N0']['delta_map'])
print('MVP v25 не изменён. Исходная validation не оценивалась.')
display(Markdown((context['output'] / 'REPORT.md').read_text()))


Этап 5/5 — итоговый отчёт | время Run All: 112.5 мин
Выбранные updates: {'N0_target_extra': '800', 'N1_nive_mixed': '0'}
Разница N1 − N0: -0.00026225630392306964
MVP v25 не изменён. Исходная validation не оценивалась.


# v32 — N0/N1: mixed NiVe pilot

Статус: complete. Исходная validation не оценивалась; v25 не изменён.

| Модель | Выбранный update | Mean raw mAP | Mean fixed-graph mAP |
|---|---:|---:|---:|
| N_ref | 0 | 0.832842 | 0.844099 |
| N0_target_extra | 800 | 0.833170 | 0.844361 |
| 50% N_ref + 50% N0_target_extra | 800 | 0.833888 | 0.845909 |
| N1_nive_mixed | 0 | 0.832842 | 0.844099 |
| 50% N_ref + 50% N1_nive_mixed | 0 | 0.832842 | 0.844099 |

Рекомендация screening: **stop**, не решение о релизе.

Все назначенные checkpoint:

| Arm | Update | Raw | Graph |
|---|---:|---:|---:|
| N0_target_extra | 0 | 0.832842 | 0.844099 |
| N0_target_extra | 400 | 0.833500 | 0.841344 |
| N0_target_extra | 800 | 0.833170 | 0.844361 |
| N0_target_extra | 1200 | 0.831079 | 0.842637 |
| N0_target_extra | 1600 | 0.832107 | 0.840464 |
| N0_target_extra | 1800 | 0.831284 | 0.840066 |
| N1_nive_mixed | 0 | 0.832842 | 0.844099 |
| N1_nive_mixed | 400 | 0.812937 | 0.826124 |
| N1_nive_mixed | 800 | 0.811222 | 0.822410 |
| N1_nive_mixed | 1200 | 0.807804 | 0.823739 |
| N1_nive_mixed | 1600 | 0.806673 | 0.821080 |
| N1_nive_mixed | 1800 | 0.821663 | 0.831807 |

## Бюджет и ограничения

1800 updates на ветвь: 1600 joint + 200 target-only.
108800 логических предъявлений на ветвь; N1: 57600 organizer + 51200 NiVe; N0: 108800 organizer.
Clean/robust дают 217600 image-forwards, а не удвоенное число уникальных фото.
Aux-loss .25 не означает 25% градиента; реальные нормы backbone-градиентов записаны каждые 25 updates.
Основные organizer-пачки/аугментации одинаковы; общее target-exposure различается намеренно.
BN обновляется одинаково: aux robust, затем main robust; clean eval; tail только target в обеих ветвях.
Primary draws используют одни identity и не являются независимыми датасетами. Это адаптивный development-screening.
Frozen step/50:50 mixture не подбирают граф/порог. Active v25 не является inner-контролем (его R1 видел holdout).
Negative pilot не доказывает бесполезность всех внешних данных; positive pilot не означает прирост над v25.
NiVe test/маски не обучались; bbox/evaluator не менялись. Номерная устойчивость и GPU не подтверждены.

## Покрытие NiVe

17070/17070 уникальных train-фото использовано.
Подробности: manifest.json, training/*/history.json, evaluation/*/metrics.json, frozen_selection.json, results.json.
